# Omni ChromHMM Analysis

Analysis, cross-segmentation comparison and inter-dataset summary plots.

Run the Snakemake pipeline first to produce the segmentations (`{ds}/.done`); then run
this notebook top-to-bottom:
1. **Per-segmentation analysis** — `analyze.run_analyze` / `analyze_peaks.run_analyze_peaks`
2. **Cross-segmentation comparison** — `compare.run_compare` / `compare_methods.run_compare_methods`
3. **Inter-dataset comparison & summary plots** — `compare`, `compare_out`, `summary_plots`, `emission_similarity`
4. **Results** — every plot displayed inline, grouped into five sections:
   (1) peaks number and lengths, (2) segmentation states number,
   (3) segmentation states lengths, (4) segmentation compositions,
   and (5) all other analyses.

In [ ]:
import glob
import importlib
import os
import sys
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import yaml

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Load configuration
config_path = os.path.abspath(os.path.expanduser("~/work/omni-chromhmm/config_encode.yaml"))
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")
workdir = os.path.expanduser(config.get("workdir", "."))

In [ ]:
# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)

import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset as compare_out
import display as _display_helpers
import emission_similarity
import match
import summary_plots
import utils

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_out, emission_similarity, match, summary_plots, utils,
           _display_helpers):
    importlib.reload(_m)

# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

# Parameters, mirroring the Snakefile
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
MARKS = ["H3K36me3", "H3K9me3", "H3K4me1", "H3K27ac", "H3K27me3", "H3K4me3"]
CHROMHMM_BIN = P["chromhmm_bin"]
OMNI_BIN = P["omni_bin"]
HOMER_BIN = P["homer_bin"]
MACS2_BIN = P["macs2_bin"]
NSTATES = P["n_states"]
MATCH_METHOD = "matched"
CALLER_BIN = {utils.OMNI: OMNI_BIN, utils.HOMER: HOMER_BIN, utils.MACS2: MACS2_BIN}

DO_REPLICATES = P.get("replicates", False)

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = [utils.HOMER, utils.MACS2, utils.OMNI]

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])
MARKUPS_DIR = os.path.join(workdir, "markups")

# De-novo methods compared across datasets
INTER_DS_METHODS = (
        [utils.CHROMHMM_DEFAULT]
        + [utils.method_key(c) for c in CALLERS]
)
# Joint models: one model per dataset over both of its replicates, see
# process_encode.sh. Same order as INTER_DS_METHODS, so the two lists pair up.
JOINT_METHODS = [utils.JOINT_CHROMHMM] + [utils.method_key(c, joint=True) for c in CALLERS]
CHIP_DATASETS = [d for d in DATASETS if not d.endswith("_mint")]
MINT_DATASETS = [d for d in DATASETS if d.endswith("_mint")]
REP_DATASETS = [d for d in DATASETS if DO_REPLICATES and DATASETS[d].get("replicates")]


# Path helpers, mirroring the Snakefile functions
def ds_of(folder):
    return folder.split("/")[0]


def folders_of(ds):
    fl = [ds]
    if DO_REPLICATES and DATASETS[ds].get("replicates"):
        fl += [f"{ds}/rep1", f"{ds}/rep2"]
    return fl


def ref_bed_path(ds):
    return f"{ds}/{DATASETS[ds]['ref_chromhmm']}_chromhmm.bed"


def seg_bin(path):
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN


def inter_ds_bed(folder, method):
    """Matched segmentation of a de-novo method in a dataset or replicate folder.

    A joint model is not per-folder: one KMeans / ChromHMM model covers both
    replicates of a dataset and writes its own segmentation per replicate under
    {ds}/joint_kmeans and {ds}/joint_chromhmm.
    """
    ds = ds_of(folder)
    sfx = MATCH_METHOD
    if method == utils.CHROMHMM_DEFAULT:
        cell = DATASETS[ds]["cell"]
        return f"{folder}/{utils.CHROMHMM_DEFAULT}_result/{cell}_{NSTATES}_dense_{sfx}.bed"
    if method.startswith("joint_"):
        rep = folder.split("/")[-1]
        if method == utils.JOINT_CHROMHMM:
            return f"{ds}/{utils.JOINT_CHROMHMM}/{rep}_{NSTATES}_dense_{sfx}.bed"
        caller = method.replace("joint_kmeans_", "")
        return f"{ds}/joint_kmeans/{caller}/{rep}_kmeans_joint_states_{sfx}.bed"
    caller = method.replace("kmeans_", "")  # omni | homer | macs2
    return f"{folder}/{caller}/{caller}_kmeans_states_{sfx}.bed"


def existing(paths):
    """Keep only paths that exist on disk (skip segmentations not produced)."""
    return [p for p in paths if os.path.exists(p)]


def needs_enrichment(outdir, cfg):
    """True when run_analyze has to (re)run for the enrichment of *outdir*.

    Either it was never produced, or the dataset has RNA-seq and the enrichment
    predates the expressed/non-expressed gene body annotations.
    """
    path = os.path.join(outdir, "enrichment", "enrichment.tsv")
    if not os.path.exists(path):
        return True
    if not cfg.get("rnaseq"):
        return False
    with open(path) as f:
        return "NonExpressedGeneBodies" not in f.read()


def rnaseq_args(ds, cfg):
    """run_analyze arguments enabling the RNA-seq annotations, for datasets that have them."""
    if not cfg.get("rnaseq"):
        return {"rnaseq": None, "gtf": None}
    return {"rnaseq": f"{ds}/rnaseq_{cfg['rnaseq']}.tsv", "gtf": GENCODE_GTF}


def comparison_present(outdir):
    """True when run_compare has already written its all-pairs table to *outdir*."""
    return (os.path.exists(os.path.join(outdir, "comparison_all_pairs.tsv")) and
            os.path.exists(os.path.join(outdir, "per_state_metrics.tsv")))


print(f"Callers       : {CALLERS}")
print(f"Match variant : {MATCH_METHOD}")
print(f"Inter methods : {INTER_DS_METHODS}")

# Metadata for plots and display
VARIANT = MATCH_METHOD  # default match variant, e.g. "comb"
SP = "out/summary_plots"  # cross-dataset summary plots
REF = "out/reference"  # ENCODE reference plots

DS_TITLE = {
    "imr90": "IMR90 (ChIP-seq)",
    "monocytes": "Monocytes (ChIP-seq)",
    "monocytes_mint": "Monocytes (Mint-ChIP)",
    "gm12878_mint": "GM12878 (Mint-ChIP)",
    "spleen": "Spleen (ChIP-seq)",
}

# Per-method segmentations shown in per-dataset grids (de-novo + reference).
METHOD_LABELS = [
    ("ref", "ENCODE reference"),
    (utils.CHROMHMM_DEFAULT, utils.display_name(utils.CHROMHMM_DEFAULT)),
    (utils.KMEANS_OMNI, utils.display_name(utils.KMEANS_OMNI)),
    (utils.KMEANS_HOMER, utils.display_name(utils.KMEANS_HOMER)),
    (utils.KMEANS_MACS2, utils.display_name(utils.KMEANS_MACS2)),
]


# 1. Performing computations


## Run Analysis
Executing the analysis for all datasets and folders.


In [ ]:
annotations = sorted(glob.glob(os.path.join(COORDS_DIR, "*.bed.gz")))
if not annotations:
    print(f"WARNING: No standard annotations found in {COORDS_DIR}!")
    annotations = sorted(glob.glob(os.path.join(TOOLS["coords_dir"], "*.bed.gz")))
    if annotations:
        print(f"Found {len(annotations)} annotations using relative path.")

futures = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    for ds, cfg in DATASETS.items():
        cell = cfg["cell"]
        folders = [ds]
        if DO_REPLICATES and cfg.get("replicates"):
            folders += [f"{ds}/rep1", f"{ds}/rep2"]

        # Peak analysis
        peaks_outdir = f"{ds}/peaks"
        if not os.path.exists(os.path.join(peaks_outdir, "peak_stats.tsv")):
            print(f"Analyzing peaks for {ds}...")
            futures.append(
                executor.submit(
                    analyze_peaks.run_analyze_peaks,
                    ds=ds, cell=cell, marks=list(MARKS), outdir=peaks_outdir,
                    omni_bin=P["omni_bin"], chromhmm_bin=CHROMHMM_BIN,
                )
            )

        # RNA-seq / ATAC extra annotations if available
        ds_annotations = list(annotations)
        if cfg.get("atac"):
            ds_annotations.append(f"{ds}/atac_{cfg['atac']}.bed.gz")

        # Segmentations analysis
        for folder in folders:
            # Reference
            ref_bed = f"{ds}/{cfg['ref_chromhmm']}_chromhmm.bed"
            ref_outdir = f"out/{folder}/ref"
            if needs_enrichment(ref_outdir, cfg):
                print(f"Analyzing reference segmentations in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=ref_bed, bin_size=CHROMHMM_BIN,
                        outdir=ref_outdir,
                        inputs=None,
                        annotations=ds_annotations,
                        bw_emissions=ref_bed.replace(".bed", ".bw_emissions.npz"),
                        **rnaseq_args(ds, cfg),
                        emissions_only=False,
                    )
                )

            # Default ChromHMM
            default_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense.bed"
            default_outdir = f"out/{folder}/chromhmm_default_dense"
            if not os.path.exists(os.path.join(default_outdir, "report.tsv")):
                print(f"Analyzing default ChromHMM in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=default_seg, bin_size=CHROMHMM_BIN,
                        outdir=default_outdir,
                        inputs=[f"{folder}/chromhmm_default/*.txt"],
                        annotations=None,
                        bw_emissions=default_seg.replace(".bed", ".bw_emissions.npz"),
                        rnaseq=None,
                        gtf=None,
                        emissions_only=False,
                    )
                )

            variant = "matched"
            matched_seg = f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_matched.bed"
            matched_outdir = f"out/{folder}/matched/chromhmm_default"
            if needs_enrichment(matched_outdir, cfg):
                print(f"Analyzing matched ChromHMM in {folder}...")
                futures.append(
                    executor.submit(
                        analyze.run_analyze,
                        seg=matched_seg, bin_size=CHROMHMM_BIN,
                        outdir=matched_outdir,
                        inputs=[f"{folder}/chromhmm_default/*.txt"],
                        annotations=ds_annotations,
                        bw_emissions=matched_seg.replace(".bed", ".bw_emissions.npz"),
                        **rnaseq_args(ds, cfg),
                        emissions_only=False,
                    )
                )

            for caller in CALLERS:
                cbin = CALLER_BIN[caller]
                peaks_dir = f"{folder}/{caller}/chromhmm_peaks"

                # KMeans
                kmeans_seg = f"{folder}/{caller}/{caller}_kmeans_states_matched.bed"
                kmeans_outdir = f"out/{folder}/matched/kmeans_{caller}"
                if needs_enrichment(kmeans_outdir, cfg):
                    print(f"Analyzing KMeans {caller} in {folder}...")
                    futures.append(
                        executor.submit(
                            analyze.run_analyze,
                            seg=kmeans_seg, bin_size=cbin,
                            outdir=kmeans_outdir,
                            inputs=[f"{peaks_dir}/chr*.txt.gz"],
                            annotations=ds_annotations,
                            bw_emissions=kmeans_seg.replace(".bed", ".bw_emissions.npz"),
                            **rnaseq_args(ds, cfg)
                        )
                    )

for fut in as_completed(futures):
    try:
        fut.result()
    except Exception as e:
        print(f"  ERROR: {e}")

print('Done')


## Cross-segmentation comparison

Per dataset: transition-matrix entropy, pairwise Cohen's
κ and Jaccard similarity, emission similarity and segment-length statistics, plus the
unified method comparison table. Requires the per-segmentation analysis above to have
run (it reads out/<dataset>/<variant>/.../jaccard.tsv and the .bin_emissions.npz files).


In [ ]:
# Cross-segmentation comparison per dataset
def compare_beds_for_folder(folder):
    cell = DATASETS[ds_of(folder)]["cell"]
    beds = [f"{folder}/chromhmm_default_result/{cell}_{NSTATES}_dense_matched.bed"]
    for caller in CALLERS:
        beds.append(f"{folder}/{caller}/{caller}_kmeans_states_matched.bed")
    return beds


def ds_compare_segs(ds):
    segs = [ref_bed_path(ds)]
    for folder in folders_of(ds):
        segs += compare_beds_for_folder(folder)
    return segs


variant = MATCH_METHOD
dataset_args = []
for ds in DATASETS:
    segs = existing(ds_compare_segs(ds))
    if len(segs) < 2:
        print(f"{ds}: found {len(segs)} segmentation(s) for '{variant}', skipping comparison")
        continue
    bins = [seg_bin(p) for p in segs]
    dataset_args.append((ds, segs, bins))

print(f"Comparing {len(dataset_args)} datasets (variant={variant}) ...")

# 1. Atomic run_compare (cross-segmentation similarity metrics & stats)
# Note: run_compare internally parallelizes pair comparisons.
for ds, segs, bins in dataset_args:
    comp_outdir = f"out/{ds}/{variant}"
    if comparison_present(comp_outdir):
        print(f"  {ds} comparison already present, skipping.")
        continue
    print(f"  {ds} ...")
    try:
        compare.run_compare(seg=segs, bins=bins,
                            outdir=comp_outdir,
                            analysis_dir=f"out/{ds}/{variant}")
    except Exception as e:
        print(f"  ERROR comparing {ds}: {e}")

# 2. Atomic run_compare_methods (aggregate results & per-method plots)
for ds, segs, bins in dataset_args:
    methods_outdir = f"out/{ds}/{variant}"
    table_path = os.path.join(methods_outdir, "comparison_table.tsv")
    if os.path.exists(table_path):
        # Check if the table is up-to-date with new ATAC metrics
        try:
            cols = pd.read_csv(table_path, sep="\t", nrows=0).columns
            if "enrich_Active_ATAC" in cols and "enrich_Active_NonExpGeneBodies" in cols:
                print(f"  {ds} compare methods already present, skipping.")
                continue
        except Exception:
            pass
    print(f"  {ds} compare methods ...")
    try:
        compare_methods.run_compare_methods(
                        analysis_dir=f"out/{ds}/{variant}",
                        comparison_dir=f"out/{ds}/{variant}",
                        outdir=methods_outdir,
                        ref_dir=f"out/{ds}")
    except Exception as e:
        print(f"  ERROR compare_methods for {ds}: {e}")


## Inter-dataset comparison


In [ ]:
# Inter-dataset comparison
ds_list = list(DATASETS)
cells = [DATASETS[d]["cell"] for d in ds_list]
sp_out = "out/summary_plots"
os.makedirs(sp_out, exist_ok=True)


def _try(label, fn):
    """Run one plotting step; report and continue on failure (e.g. missing inputs)."""
    try:
        fn()
    except Exception as e:
        print(f"  SKIP {label}: {e}")


# 1. Per-method cross-dataset comparison (every dataset pair).
for method in INTER_DS_METHODS:
    pairs = [(d, inter_ds_bed(d, method)) for d in ds_list]
    pairs = [(d, p) for d, p in pairs if os.path.exists(p)]
    if len(pairs) < 2:
        print(f"  SKIP inter compare {method}: <2 datasets with this segmentation")
        continue

    outdir = f"out/{method}"
    if comparison_present(outdir):
        print(f"  {method} inter-dataset comparison already present, skipping.")
        continue

    segs = [p for _, p in pairs]
    labels = [f"{d}:{method}" for d, _ in pairs]
    bins = [seg_bin(p) for p in segs]
    print(f"Inter-dataset compare: {method} ({len(segs)} datasets)")
    _try(f"compare {method}",
         lambda segs=segs, bins=bins, labels=labels, method=method, outdir=outdir:
         compare.run_compare(seg=segs, bins=bins, labels=labels, all_pairs=True,
                             outdir=outdir))

# 2. Aggregate per-method kappa matrices into one cross-dataset table.
if not os.path.exists("out/comparison_table.tsv"):
    _try("comparison_table", lambda: compare_out.run_compare_out(
        methods=INTER_DS_METHODS, indir="out",
        outfile="out/comparison_table.tsv"))


In [ ]:
# Pairwise similarity among all ENCODE reference segmentations + reference plots.
ref_segs = sorted(glob.glob(os.path.join(MARKUPS_DIR, "15state", "*.bed.gz")),
                  key=lambda p: "_".join(os.path.basename(p).replace(".bed.gz", "").split("_")[1:]))
if not ref_segs:
    print("No reference markups in", os.path.join(MARKUPS_DIR, "15state"))
else:
    ref_outdir = "out/reference"
    if comparison_present(ref_outdir):
        print("Reference comparison already present, skipping.")
    else:
        ref_labels = ["_".join(os.path.basename(p).replace(".bed.gz", "").split("_")[1:])
                      for p in ref_segs]
        _try("reference compare", lambda: compare.run_compare(
            seg=ref_segs, bins=CHROMHMM_BIN, labels=ref_labels, all_pairs=True,
            outdir=ref_outdir))


### Optimal Number of States Analysis (Computation)
1. **Elbow Method (Inertia)**: Look for the point where the rate of decrease in inertia significantly slows down.
2. **Silhouette Score**: Higher average silhouette scores indicate better-defined clusters.
3. **Transition Matrix Entropy**: Lower entropy indicates more predictable transitions between states.

In [ ]:
# Silhouette and Elbow analysis for all datasets (Computation)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import subprocess

n_states_range = range(2, 71)
sample_size = 10_000
n_states_results_path = "out/optimal_n_states.tsv"
# ! rm -f {n_states_results_path}

# Each (dataset, method) sweep is cached on its own, so a rerun only fills gaps.
os.makedirs("out/n_states", exist_ok=True)
SWEEP_COLUMNS = ["inertia", "silhouette", "entropy", "actual_n_states"]


class MissingInput(Exception):
    """Raised by a sweep whose binarized chr22 input is not on disk."""


def sweep_cache_path(ds, method):
    return f"out/n_states/{ds}_{method}.tsv"


def sweep_df(**columns):
    """A sweep as one row per n_states; metrics a method does not have are NaN."""
    blank = [np.nan] * len(n_states_range)
    return pd.DataFrame({"n_states": list(n_states_range),
                         **{c: columns.get(c, blank) for c in SWEEP_COLUMNS}})


def sweep_valid(df):
    """A cached sweep is reusable once every n_states has an entropy."""
    return ("n_states" in df.columns and "entropy" in df.columns
            and set(df["n_states"]) == set(n_states_range)
            and not df["entropy"].isna().any())


def sweep_series(df):
    """The {metric: values by n_states} view of a sweep used by the plots below."""
    df = df.sort_values("n_states")
    return {c: df[c].tolist() if c in df.columns else [np.nan] * len(df)
            for c in SWEEP_COLUMNS}


# Helper for entropy
def compute_entropy_from_labels(labels, bin_size):
    # labels is a numpy array of state indices
    segs = []
    for i, label in enumerate(labels):
        segs.append(["chr22", i * bin_size, (i + 1) * bin_size, str(label)])
    states, counts, state_bp = analyze.build_transition_matrix(segs, bin_size)
    if not states:
        return 0
    total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
    return total_H


def kmeans_sweep(ds, caller):
    """KMeans on the chr22 binarization of *caller*, for every n in n_states_range."""
    cell = DATASETS[ds]["cell"]
    paths_to_try = [
        os.path.join(ds, caller, "chromhmm_peaks", f"{cell}_chr22_binary.txt.gz"),
        os.path.join(ds, caller, "chromhmm_peaks", f"{ds}_chr22_binary.txt.gz"),
        os.path.join(ds, caller, "chromhmm_peaks", f"chr22_binary.txt.gz")
    ]
    data_path = next((p for p in paths_to_try if os.path.exists(p)), None)
    if not data_path:
        raise MissingInput(f"no chr22 binarization for {ds} {caller}")

    print(f"Loading data for {ds} {caller} from {data_path}...")
    chrom, marks, X = analyze.load_binary(data_path)
    bin_size = P.get(f"{caller}_bin", 200)

    if X.shape[0] > sample_size:
        np.random.seed(42)
        idx = np.random.choice(X.shape[0], sample_size, replace=False)
        X_sample = X[idx]
    else:
        X_sample = X

    inertia, silhouette_scores, entropies, actual_n_states = [], [], [], []
    base_inertia = max(1, np.sum((X - X.mean(axis=0))**2))

    for n in n_states_range:
        kmeans = KMeans(n_clusters=n, init='k-means++', random_state=42, n_init=10)
        kmeans.fit(X)
        inertia.append(kmeans.inertia_ / base_inertia)
        sample_labels = kmeans.predict(X_sample)
        silhouette_scores.append(silhouette_score(X_sample, sample_labels))
        entropies.append(compute_entropy_from_labels(kmeans.labels_, bin_size))
        actual_n_states.append(len(np.unique(kmeans.labels_)))

    return sweep_df(inertia=inertia, silhouette=silhouette_scores,
                    entropy=entropies, actual_n_states=actual_n_states)


def chromhmm_sweep(ds):
    """ChromHMM LearnModel on chr22, for every n in n_states_range."""
    cell = DATASETS[ds]["cell"]
    chr22_indir = os.path.join(ds, "chromhmm_chr22")
    os.makedirs(chr22_indir, exist_ok=True)
    src = None
    for indir in [os.path.join(ds, "chromhmm_default")] + [os.path.join(ds, c, "chromhmm_peaks") for c in CALLERS]:
        if os.path.exists(indir):
            for f in os.listdir(indir):
                if (f.endswith("chr22_binary.txt.gz") or f.endswith("chr22_binary.txt")) and (cell in f or ds in f):
                    src = os.path.join(indir, f)
                    break
        if src: break

    if not src:
        raise MissingInput(f"no chr22 binarization for {ds} chromhmm")

    dst = os.path.join(chr22_indir, os.path.basename(src))
    src_abs = os.path.abspath(src)
    # A link left by an earlier run can point at a moved workdir: broken, so os.path.exists
    # says False, yet os.symlink still fails because the link itself is there. Re-point it.
    if os.path.islink(dst) and os.path.realpath(dst) != src_abs:
        os.unlink(dst)
    if not os.path.lexists(dst):
        os.symlink(src_abs, dst)

    entropies, actual_n_states = [], []
    for n in n_states_range:
        outdir = os.path.join(ds, f"chromhmm_chr22_res_{n}")
        seg_path = os.path.join(outdir, f"{cell}_{n}_segments.bed")
        if not os.path.exists(seg_path):
            cmd = ["java", "-mx4000M", "-jar", os.path.join(workdir, config["tools"]["chromhmm_jar"]),
                   "LearnModel", "-b", str(CHROMHMM_BIN), chr22_indir, outdir, str(n), config["params"]["genome"]]
            print(f"Running ChromHMM for {ds} n={n}...")
            try:
                subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except subprocess.CalledProcessError: pass

        if os.path.exists(seg_path):
            segs = analyze.load_bed(seg_path)
            states, counts, state_bp = analyze.build_transition_matrix(segs, 200)
            total_H, _, _, _ = analyze.transition_entropy(states, counts, state_bp)
            entropies.append(total_H)
            actual_n_states.append(len(states))
        else:
            entropies.append(np.nan)
            actual_n_states.append(np.nan)

    return sweep_df(entropy=entropies, actual_n_states=actual_n_states)


def compute_optimal_n_states():
    """Every dataset and method swept over n_states_range, one row per n_states."""
    frames = []
    for ds in DATASETS:
        sweeps = [(caller, lambda ds=ds, caller=caller: kmeans_sweep(ds, caller))
                  for caller in CALLERS]
        sweeps.append((utils.CHROMHMM, lambda ds=ds: chromhmm_sweep(ds)))
        for method, sweep in sweeps:
            try:
                df = utils.cached_csv(sweep_cache_path(ds, method), sweep, sep="\t",
                                      label=f"{ds} {method} n_states sweep",
                                      valid=sweep_valid)
            except MissingInput as e:
                print(f"Skipping {ds} {method}: {e}")
                continue
            frames.append(df.assign(dataset=ds, method=method))
    if not frames:
        return pd.DataFrame(columns=["dataset", "method", "n_states"] + SWEEP_COLUMNS)
    return pd.concat(frames, ignore_index=True)[["dataset", "method", "n_states"] + SWEEP_COLUMNS]


def n_states_valid(df):
    """The combined table is reusable while every method holds a complete sweep."""
    if df.empty or "dataset" not in df.columns or "method" not in df.columns:
        return False
    return all(sweep_valid(group) for _, group in df.groupby(["dataset", "method"]))


df_results = utils.cached_csv(n_states_results_path, compute_optimal_n_states,
                              label="optimal n_states results", sep="\t", valid=n_states_valid)

n_states_results = {key: sweep_series(group)
                    for key, group in df_results.groupby(["dataset", "method"])}

# 2. Doing plotting


In [ ]:
# 2. Doing plotting
_try("reference summary plots", lambda: summary_plots.run_summary_plots(
        markups_dir=MARKUPS_DIR,
        ref_composition_outfile="out/reference/state_composition.png",
        ref_comp_matrix="out/reference/composition_similarity_matrix.tsv",
        ref_kappa_matrix="out/reference/kappa_matrix.tsv",
        ref_jaccard_matrix="out/reference/jaccard_similarity_matrix.tsv",
        ref_dist_outfile="out/reference/similarity_distribution.png",
        ref_comp_noqh_matrix="out/reference/composition_noqh_similarity_matrix.tsv",
        ref_kappa_noqh_matrix="out/reference/kappa_noqh_matrix.tsv",
        ref_jaccard_noqh_matrix="out/reference/jaccard_noqh_matrix.tsv",
        ref_dist_noqh_outfile="out/reference/similarity_distribution_noqh.png"))


In [ ]:
# Cross-dataset summary bar / distribution plots.
methods_dirs = [f"out/{d}/{MATCH_METHOD}" for d in ds_list]
analysis_dirs = [f"out/{d}/{MATCH_METHOD}" for d in ds_list]

_try("summary bar plots", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, methods_dirs=methods_dirs, analysis_dirs=analysis_dirs,
    methods=INTER_DS_METHODS, outdir=sp_out))

_try("state coverage", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    state_coverage_outfile=f"{sp_out}/state_coverage.png"))

_try("peak stats", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, workdir=workdir, methods=INTER_DS_METHODS,
    peak_count_outfile=f"{sp_out}/n_peaks.png",
    peak_length_outfile=f"{sp_out}/peak_length.png"))
_try("method similarity distribution", lambda: summary_plots.run_summary_plots(
    method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
    method_sim_dist_outfile=f"{sp_out}/method_similarity_distribution.png",
    method_sim_dist_noqh_outfile=f"{sp_out}/method_similarity_distribution_noqh.png"))

if CHIP_DATASETS and MINT_DATASETS:
    _try("ChIP vs Mint similarity distribution", lambda: summary_plots.run_summary_plots(
        method_sim_dist_indir="out", method_sim_dist_methods=INTER_DS_METHODS,
        method_sim_dist_group_a=CHIP_DATASETS, method_sim_dist_group_b=MINT_DATASETS,
        method_sim_dist_filtered_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint.png",
        method_sim_dist_filtered_noqh_outfile=f"{sp_out}/method_similarity_distribution_chip_vs_mint_noqh.png"))

if REP_DATASETS:
    _try("replicate consistency", lambda: summary_plots.run_summary_plots(
        datasets=REP_DATASETS, methods_dirs=[f"out/{d}/{VARIANT}" for d in REP_DATASETS],
        methods=INTER_DS_METHODS, rep_consistency_outdir=sp_out))

_try("per-dataset state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, methods=INTER_DS_METHODS,
    nstates=NSTATES, match_method=MATCH_METHOD, 
    method_ds_composition_outdir=sp_out,
    all_methods_composition_outdir=sp_out))
_try("mean state composition", lambda: summary_plots.run_summary_plots(
    datasets=ds_list, cells=cells, workdir=workdir, markups_dir=MARKUPS_DIR,
    methods=INTER_DS_METHODS, nstates=NSTATES, match_method=MATCH_METHOD,
    method_composition_outfile=f"{sp_out}/method_state_composition.png"))

# Per-assay (ChIP-seq vs Mint-ChIP) splits of the cross-dataset summaries
# Regenerate the peaks, total-segments and per-dataset ENCODE-reference plots
# separately for each assay group, into chip/ and mint/ subdirs of sp_out.
for _grp, _gname, _dss in [("chip", "ChIP-seq", CHIP_DATASETS),
                           ("mint", "Mint-ChIP", MINT_DATASETS)]:
    if not _dss:
        continue
    _gdir = f"{sp_out}/{_grp}"
    os.makedirs(_gdir, exist_ok=True)
    _mdirs = [f"out/{d}/{MATCH_METHOD}" for d in _dss]
    _adirs = [f"out/{d}/{MATCH_METHOD}" for d in _dss]
    _try(f"summary bars [{_grp}]", lambda dss=_dss, md=_mdirs, ad=_adirs, gd=_gdir:
    summary_plots.run_summary_plots(datasets=dss, methods_dirs=md, analysis_dirs=ad,
                                    methods=INTER_DS_METHODS, outdir=gd,
                                    all_methods_composition_outdir=gd))
    _try(f"peaks [{_grp}]", lambda dss=_dss, gd=_gdir:
    summary_plots.run_summary_plots(datasets=dss, workdir=workdir, methods=INTER_DS_METHODS,
                                    peak_count_outfile=f"{gd}/n_peaks.png",
                                    peak_length_outfile=f"{gd}/peak_length.png"))
    _try(f"reference n_segments [{_grp}]", lambda dss=_dss, md=_mdirs, gd=_gdir, gn=_gname:
    summary_plots.plot_reference_n_segments(
        dss, md, [DATASETS[d]["cell"] for d in dss],
        f"{gd}/reference_n_segments.png", f"ENCODE reference segments — {gn}"))


In [ ]:
# Emission discriminability (Gini) per dataset + summary.
analysis_dirs_matched = [f"out/{d}/matched" for d in ds_list]
if not os.path.exists(f"{sp_out}/emission_gini_summary.png"):
    _try("emission similarity", lambda: emission_similarity.run_emission_similarity(
        datasets=ds_list, analysis_dirs=analysis_dirs_matched, methods=INTER_DS_METHODS,
        outdir=sp_out))


### Optimal Number of States Analysis (Plotting)


In [ ]:
# Optimal Number of States Analysis (Plotting)
if n_states_results:
    # Combined plots (mean with error across datasets per method)
    combined_rows = []
    _m_map = {
        utils.CHROMHMM: utils.display_name(utils.CHROMHMM_DEFAULT),
        utils.HOMER:    utils.display_name(utils.KMEANS_HOMER),
        utils.MACS2:    utils.display_name(utils.KMEANS_MACS2),
        utils.OMNI:     utils.display_name(utils.KMEANS_OMNI)
    }
    # Use global coloring for methods
    method_palette = {
        _m_map[utils.CHROMHMM]: utils.method_color(utils.CHROMHMM_DEFAULT),
        _m_map[utils.OMNI]:     utils.method_color(utils.KMEANS_OMNI),
        _m_map[utils.HOMER]:    utils.method_color(utils.KMEANS_HOMER),
        _m_map[utils.MACS2]:    utils.method_color(utils.KMEANS_MACS2),
    }
    hue_order = [_m_map[utils.CHROMHMM], _m_map[utils.HOMER], _m_map[utils.MACS2], _m_map[utils.OMNI]]
    for (ds, m), res in n_states_results.items():
        for i, n in enumerate(n_states_range):
            combined_rows.append({
                "dataset": ds,
                "method": _m_map.get(m, m),
                "n_states": n,
                "inertia": res["inertia"][i],
                "silhouette": res["silhouette"][i],
                "entropy": res["entropy"][i],
                "actual_n_states": res["actual_n_states"][i]
            })
    df_plot = pd.DataFrame(combined_rows)
    n_ds = df_plot['dataset'].nunique()

    # Combined Elbow
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['inertia'].isna()], x='n_states', y='inertia', 
                 hue='method', hue_order=hue_order, palette=method_palette, marker='o', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Normalized Inertia')
    ax.set_title(f'Elbow Method (Normalized Inertia) - Combined (n={n_ds})')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_elbow.png")
    plt.close(fig)

    # Combined Silhouette
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['silhouette'].isna()], x='n_states', y='silhouette', 
                 hue='method', hue_order=hue_order, palette=method_palette, marker='s', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Silhouette Score')
    ax.set_title(f'Silhouette Score - Combined (n={n_ds})')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_silhouette.png")
    plt.close(fig)

    # Combined Entropy
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=df_plot[~df_plot['entropy'].isna()], x='n_states', y='entropy', 
                 hue='method', hue_order=hue_order, palette=method_palette, marker='^', ax=ax)
    ax.set_xlabel('Number of States')
    ax.set_ylabel('Transition Matrix Entropy (bits)')
    ax.set_title(f'Transition Matrix Entropy - Combined (n={n_ds})')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_entropy.png")
    plt.close(fig)

    # Combined Actual Number of States
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.barplot(data=df_plot[~df_plot['actual_n_states'].isna()], x='n_states', y='actual_n_states', 
                hue='method', hue_order=hue_order, palette=method_palette, ax=ax, capsize=.1, errwidth=1)
    utils.strip_points(ax, data=df_plot[~df_plot['actual_n_states'].isna()],
                       x='n_states', y='actual_n_states', hue='method',
                       hue_order=hue_order, size=2)
    ax.set_xlabel('Requested Number of States')
    ax.set_ylabel('Actual Number of States')
    ax.set_title(f'Actual vs Requested Number of States - Combined (n={n_ds})')
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig("out/optimal_n_states_actual.png")
    plt.close(fig)

    methods = sorted(set(m for d, m in n_states_results.keys()))
    for method in methods:
        method_label = utils.OMNI_DISPLAY.lower() if method == utils.OMNI else method
        # Plotting results - Elbow Method
        fig1, ax1 = plt.subplots(figsize=(10, 6))
        plotted_elbow = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if np.isnan(res["inertia"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax1.plot(list(n_states_range), res["inertia"], marker='o', label=label)
            plotted_elbow = True

        if plotted_elbow:
            ax1.set_xlabel('Number of States')
            ax1.set_ylabel('Normalized Inertia')
            ax1.set_title(f'Elbow Method (Normalized Inertia) - {method_label}')
            ax1.grid(True, linestyle='--', alpha=0.5)
            ax1.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            os.makedirs("out", exist_ok=True)
            plt.savefig(f"out/optimal_n_states_elbow_{method_label}.png")
        plt.close(fig1)

        # Plotting results - Silhouette Score
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        plotted_silhouette = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if np.isnan(res["silhouette"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax2.plot(list(n_states_range), res["silhouette"], marker='s', label=label)
            plotted_silhouette = True

        if plotted_silhouette:
            ax2.set_xlabel('Number of States')
            ax2.set_ylabel('Silhouette Score')
            ax2.set_title(f'Silhouette Score - {method_label}')
            ax2.grid(True, linestyle='--', alpha=0.5)
            ax2.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            plt.savefig(f"out/optimal_n_states_silhouette_{method_label}.png")
        plt.close(fig2)

        # Plotting results - Transition Matrix Entropy
        fig3, ax3 = plt.subplots(figsize=(10, 6))
        plotted_entropy = False
        for (ds, m), res in n_states_results.items():
            if m != method: continue
            if "entropy" not in res or np.isnan(res["entropy"]).all(): continue
            label = f"{DS_TITLE.get(ds, ds)}"
            ax3.plot(list(n_states_range), res["entropy"], marker='^', label=label)
            plotted_entropy = True

        if plotted_entropy:
            ax3.set_xlabel('Number of States')
            ax3.set_ylabel('Transition Matrix Entropy (bits)')
            ax3.set_title(f'Transition Matrix Entropy - {method_label}')
            ax3.grid(True, linestyle='--', alpha=0.5)
            ax3.legend(fontsize='small', ncol=2)
            plt.tight_layout()
            plt.savefig(f"out/optimal_n_states_entropy_{method_label}.png")
        plt.close(fig3)


# 3. Showing the results

# Results
 Every plot produced by the computation cells above, displayed inline and
organised into five sections:

1. **Peaks — number and lengths**
2. **Segmentation — number of states and segments**
3. **Segmentation — state lengths**
4. **Segmentation — state composition**
5. **All other analyses** — entropy, similarity, biological validation,
   emission discriminability, cross-assay portability, replicate consistency
   and per-dataset detail.

Run the helper cell first; missing files are skipped silently, so the output
reflects exactly the segmentations that were produced.

In [ ]:
# Display helpers. Every plot below was produced by the computation cells above
# and is read straight off disk. Missing files are skipped silently, so the
# notebook renders cleanly regardless of which segmentations were produced.
# Shared with the other analysis notebooks: scripts/analysis/display.py
from display import header, method_plot, show, show_group, show_table

print("Display helpers ready (variant:", VARIANT + ").")

## 1. Peaks — number and lengths

Binarization peak statistics: peak count, mean peak length and gap lengths
between adjacent binarized elements, summarised across datasets and shown per
dataset (including replicate Jaccard where replicates exist).

In [ ]:
# 1. Peaks — number and lengths
show_group("Cross-dataset summary", [
    (f"{SP}/n_peaks.png", "Peak count per mark/method — mean \u00b1 std across datasets"),
    (f"{SP}/peak_length.png", "Mean peak length per mark/method — mean \u00b1 std across datasets"),
    (f"{SP}/per_state_kappa_bar.png", "Per-state Kappa vs ENCODE reference — combined across methods"),
    (f"{SP}/per_state_jaccard_bar.png", "Per-state Jaccard vs ENCODE reference — combined across methods"),
], level=2)

for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/n_peaks.png", "Peak count per mark/method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/peak_length.png", "Mean peak length per mark/method — mean \u00b1 std across datasets"),
        (f"{SP}/{_g}/per_state_kappa_bar.png", "Per-state Kappa vs ENCODE reference — combined across methods"),
        (f"{SP}/{_g}/per_state_jaccard_bar.png", "Per-state Jaccard vs ENCODE reference — combined across methods"),
    ], level=2)

## 2. Segmentation — number of states and segments

How fragmented each segmentation is: the total segment count per method across
datasets, the per-reference state/segment counts, and per-dataset
state/segment counts for every segmentation.

In [ ]:
# 2. Segmentation — number of states and segments
for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"Cross-dataset summary — {_gname}", [
        (f"{SP}/{_g}/summary_n_segments.png",
         "Total number of segments per method (incl. ENCODE reference) — mean \u00b1 std across datasets"),
    ], level=2)

for _g, _gname in [("chip", "ChIP-seq"), ("mint", "Mint-ChIP")]:
    show_group(f"ENCODE reference segmentations — {_gname}", [
        (f"{SP}/{_g}/reference_n_segments.png", "Number of segments per dataset's ENCODE reference"),
    ])


## 3. Segmentation — state lengths

Segment length distributions: the cross-dataset per-state comparison and
coverage, the ENCODE reference length summaries, and per-dataset segment-length
statistics with per-method length distributions.

In [ ]:
# 3. Segmentation — state lengths
show_group("Cross-dataset summary", [
    (f"{SP}/state_coverage.png", "Genomic coverage fraction per chromatin state"),
    (f"{SP}/summary_mean_tx_length.png", "Mean Tx (transcription) segment length"),
], level=2)

show_group("ENCODE reference segmentations", [
    (f"{REF}/mean_length.png", "Mean segment length"),
    (f"{REF}/median_length.png", "Median segment length"),
    (f"{REF}/min_length.png", "Min segment length"),
    (f"{REF}/max_length.png", "Max segment length"),
], width=750)


## 4. Segmentation — state composition

Fraction of the genome covered by each chromatin state: averaged per method
across datasets, across the ENCODE reference segmentations, and per dataset for
each de-novo method.

In [ ]:
# 4. Segmentation — state composition
show_group("Per-method composition (mean across datasets)", [
    (f"{SP}/method_state_composition.png", "State composition per method — mean across datasets"),
], level=2)

show_group("ENCODE reference composition", [
    (f"{REF}/state_composition.png", "State composition across ENCODE reference segmentations"),
])

comp = [(p, os.path.basename(p).replace("method_ds_composition_", "").replace(".png", ""))
        for p in sorted(glob.glob(f"{SP}/method_ds_composition_*.png"))]
show_group("Per-dataset composition for each de-novo method", comp)

comp2 = [(p, os.path.basename(p).replace("ds_composition_", "").replace(".png", ""))
         for p in sorted(glob.glob(f"{SP}/ds_composition_*.png"))]
show_group("Per-method composition for each dataset", comp2)


## 5. All other analyses

Transition entropy, inter-dataset / inter-reference similarity, biological
validation (RNA-seq / ATAC-seq), emission discriminability, cross-assay
portability, replicate consistency, the aggregated comparison table, and the
per-dataset detail (method comparison tables, functional enrichment and state
emissions for each method).

In [ ]:
# 5. All other analyses
show_group("Transition matrix entropy", [
    (f"{SP}/summary_entropy.png", "De-novo — Full (raw labels)"),
    (f"{SP}/summary_entropy_noqh.png", "De-novo — NOQH (excl. Quies/Het)"),
    (f"{REF}/entropy_summary_combined.png", "ENCODE reference entropy (Full + NOQH)"),
], level=2)

show_group("Segmentation similarity", [
    (f"{SP}/method_similarity_distribution.png", "De-novo inter-dataset similarity (3 variants) — Full"),
    (f"{SP}/method_similarity_distribution_noqh.png", "De-novo inter-dataset similarity (3 variants) — NOQH"),
    (f"{SP}/rep_consistency_distribution.png", "De-novo replicate consistency (3 variants) — Full"),
    (f"{SP}/rep_consistency_distribution_noqh.png", "De-novo replicate consistency (3 variants) — NOQH"),
    (f"{SP}/rep_consistency_per_state_jaccard.png", "Per-state replicate consistency: Jaccard"),
    (f"{SP}/rep_consistency_per_state_kappa.png", "Per-state replicate consistency: Cohen's Kappa"),
    (f"{REF}/similarity_distribution.png", "Inter-reference similarity (3 variants) — Full"),
    (f"{REF}/similarity_distribution_noqh.png", "Inter-reference similarity (3 variants) — NOQH"),
    (f"{SP}/summary_kappa_vs_ref.png", "Kappa agreement vs ENCODE reference"),
    (f"{SP}/summary_jaccard_vs_ref.png", "Jaccard agreement vs ENCODE reference"),
    (f"{SP}/per_state_kappa_bar.png", "Per-state Kappa vs ENCODE reference — combined across methods"),
    (f"{SP}/per_state_jaccard_bar.png", "Per-state Jaccard vs ENCODE reference — combined across methods"),
    (f"{SP}/per_state_kappa_summary.png", "Per-state Kappa vs ENCODE reference (Summary)"),
    (f"{SP}/per_state_jaccard_summary.png", "Per-state Jaccard vs ENCODE reference (Summary)"),
], level=2)

show_group("Biological validation (RNA-seq / ATAC-seq)", [
    (f"{SP}/summary_jaccard_tx.png", "Jaccard: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_enrich_tx.png", "Tx fold enrichment at expressed gene bodies"),
    (f"{SP}/summary_sensitivity_tx.png", "Fraction of expressed gene bodies covered by Tx states"),
    (f"{SP}/summary_coverage_tx.png", "Fraction of Tx states covered by expressed genes"),
    (f"{SP}/summary_2way_tx.png", "2-way validation: Tx state vs expressed gene bodies"),
    (f"{SP}/summary_jaccard_tss.png", "Jaccard: Tss state vs RefSeq TSS ±2 kb"),
    (f"{SP}/summary_enrich_tss.png", "Tss fold enrichment at RefSeq TSS ±2 kb"),
    (f"{SP}/summary_sensitivity_tss.png", "Fraction of RefSeq TSS ±2 kb covered by Tss states"),
    (f"{SP}/summary_coverage_tss.png", "Fraction of Tss states covered by RefSeq TSS ±2 kb"),
    (f"{SP}/summary_2way_tss.png", "2-way validation: Tss state vs RefSeq TSS ±2 kb"),
    (f"{SP}/summary_jaccard_tss_exptss.png", "Jaccard: Tss state vs Expressed TSS"),
    (f"{SP}/summary_jaccard_tss_exptss2kb.png", "Jaccard: Tss state vs Expressed TSS ±2 kb"),
    (f"{SP}/summary_enrich_tss_exptss2kb.png", "Tss fold enrichment at Expressed TSS ±2 kb"),
    (f"{SP}/summary_sensitivity_tss_exptss.png", "Fraction of Expressed TSS covered by Tss states"),
    (f"{SP}/summary_coverage_tss_exptss.png", "Fraction of Tss states covered by Expressed TSS"),
    (f"{SP}/summary_2way_tss_exptss.png", "2-way validation: Tss state vs Expressed TSS"),
    (f"{SP}/summary_2way_tss_exptss2kb.png", "2-way validation: Tss state vs Expressed TSS ±2 kb"),
    (f"{SP}/summary_jaccard_active_atac.png", "Jaccard: Active states vs ATAC-seq"),
    (f"{SP}/summary_enrich_active_atac.png", "Active chromatin enrichment at ATAC-seq peaks"),
    (f"{SP}/summary_sensitivity_active_atac.png", "Fraction of ATAC-seq peaks covered by Active states"),
    (f"{SP}/summary_coverage_active_atac.png", "Fraction of Active states covered by ATAC-seq peaks"),
    (f"{SP}/summary_2way_active_atac.png", "2-way validation: Active chromatin vs ATAC-seq"),
    (f"{SP}/summary_sensitivity_tss_atac.png", "Fraction of ATAC-seq peaks covered by Tss states"),
    (f"{SP}/summary_sensitivity_enh_atac.png", "Fraction of ATAC-seq peaks covered by Enh states"),
    (f"{SP}/summary_enrich_quies_atac.png", "Quiescent states enrichment at ATAC-seq peaks (depletion)"),
    (f"{SP}/summary_enrich_active_nonexp.png", "Active states enrichment at Non-expressed Gene Bodies (depletion)"),
    (f"{SP}/summary_enrich_quies_nonexp.png", "Quiescent states enrichment at Non-expressed Gene Bodies"),
], level=2)

show_group("Emission discriminability (Gini index)", [
    (f"{SP}/emission_gini_summary.png", "Gini index of state emissions"),
], level=2)

show_group("Cross-assay portability (ChIP ↔ Mint-ChIP)", [
    (f"{SP}/method_similarity_distribution_chip_vs_mint.png", utils.FULL_DISPLAY),
    (f"{SP}/method_similarity_distribution_chip_vs_mint_noqh.png", utils.NOQH_DISPLAY),
], level=2)

show_group("Replicate consistency", sorted(glob.glob(f"{SP}/rep_consistency_*.png")), level=2)

header("Cross-dataset comparison table", 2)
if not show_table("out/comparison_table.tsv"):
    print("  (no aggregated comparison table)")



## 6. Per-state matching matrices

Work-state → ENCODE-reference matching score matrices produced by `match.py`. Rows are the de-novo method's states, columns the reference states; the Hungarian-selected match per row is outlined in red. Produced by the pipeline as `{...}_matched.match.png` next to each matched BED.

In [ ]:
# Uncomment to show
# # Per-state matching matrices (work → ENCODE reference).
# header("Per-state matching matrices (work → ENCODE reference)", 2)
# _match_methods = [("chromhmm_default", "Default ChromHMM"),
#                   ("kmeans_omni", "KMeans OmniPeak"),
#                   ("kmeans_homer", "KMeans HOMER"),
#                   ("kmeans_macs2", "KMeans MACS2")]
# for ds in DATASETS:
#     items = [(inter_ds_bed(ds, k).replace(".bed", ".match.png"),
#               f"{lbl}: per-state matching score (matched cell outlined in red)")
#              for k, lbl in _match_methods]
#     show_group(DS_TITLE.get(ds, ds), items, width=620, level=3)
#     break  # Comment to plot all the dataset plots


# 7. All other per dataset plots

In [ ]:
# Uncomment to show
# for ds in DATASETS:
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"{ds}/peaks/n_peaks.png", "Number of peaks per mark"),
#         (f"{ds}/peaks/mean_length.png", "Mean peak length per mark"),
#         (f"{ds}/peaks/median_length.png", "Median peak length per mark"),
#         (f"{ds}/peaks/jaccard_rep1_vs_rep2.png", "Peak Jaccard: rep1 vs rep2"),
#     ], width=600, level=3)
#
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"out/{ds}/{VARIANT}/n_segments.png", "Number of segments per segmentation"),
#         (f"out/{ds}/{VARIANT}/mean_Tx_length.png", "Mean Tx length per segmentation"),
#         (f"{SP}/per_state_kappa_{ds}.png", "Per-state Cohen's Kappa vs ENCODE reference"),
#     ], width=750, level=3)
#
#     show_group(DS_TITLE.get(ds, ds), [
#         (f"out/{ds}/{VARIANT}/mean_length.png", "Mean segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/median_length.png", "Median segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/min_length.png", "Min segment length per segmentation"),
#         (f"out/{ds}/{VARIANT}/max_length.png", "Max segment length per segmentation"),
#     ], width=750, level=3)
#     show_group(f"{DS_TITLE.get(ds, ds)} — per-method length distribution",
#                [(method_plot(ds, k, "segment_length.png", VARIANT), lbl) for k, lbl in METHOD_LABELS],
#                width=600, level=4)
#
#     header(DS_TITLE.get(ds, ds), 3)
#     show_table(f"out/{ds}/{VARIANT}/comparison_table.tsv", caption="Method comparison table")
#     for k, lbl in METHOD_LABELS:
#         show_group(lbl, [
#             (method_plot(ds, k, "enrichment/enrichment.png", VARIANT), f"{lbl}: functional enrichment"),
#             (method_plot(ds, k, "bin_emissions/state_emissions.png", VARIANT), f"{lbl}: binarized emissions"),
#             (method_plot(ds, k, "bw_emissions/state_emissions.png", VARIANT), f"{lbl}: bigwig emissions"),
#         ], width=760, level=4)

## 8. Optimal Number of States Results

In [ ]:
if n_states_results:
    show_group("Optimal Number of States Analysis - Combined", [
        ("out/optimal_n_states_elbow.png", "Elbow Method Analysis (Mean ± SE)"),
        ("out/optimal_n_states_silhouette.png", "Silhouette Score Analysis (Mean ± SE)"),
        ("out/optimal_n_states_entropy.png", "Transition Matrix Entropy Analysis (Mean ± SE)"),
        ("out/optimal_n_states_actual.png", "Actual Number of States Analysis (Mean ± SE)")
    ], width=800, level=3)

    methods = sorted(set(m for d, m in n_states_results.keys()))
    for method in methods:
        method_label = utils.OMNI_DISPLAY.lower() if method == utils.OMNI else method
        header(f"Method: {method_label}", level=3)
        show_group(f"Optimal Number of States Analysis - {method_label}", [
            (f"out/optimal_n_states_elbow_{method_label}.png", f"Elbow Method Analysis - {method_label}"),
            (f"out/optimal_n_states_silhouette_{method_label}.png", f"Silhouette Score Analysis - {method_label}"),
            (f"out/optimal_n_states_entropy_{method_label}.png", f"Transition Matrix Entropy Analysis - {method_label}")
        ], width=800, level=4)


# Cell type differences in chromatin

### 1. Compute

In [ ]:
os.makedirs("out/consistency", exist_ok=True)

# 1. Load segmentations
method_segs = defaultdict(list)
# We want to compare across all datasets for each method
for ds in CHIP_DATASETS:
    # Reference
    try:
        ref_path = ref_bed_path(ds)
        if os.path.exists(ref_path):
            method_segs["Reference"].append(match.load_bed(ref_path))
    except Exception as e:
        print(f"Warning: could not load reference for {ds}: {e}")
    
    # De-novo methods
    _consistency_methods = [(utils.CHROMHMM_DEFAULT, utils.CHROMHMM_DISPLAY),
                            (utils.KMEANS_HOMER, utils.HOMER_DISPLAY),
                            (utils.KMEANS_MACS2, utils.MACS2_DISPLAY),
                            (utils.KMEANS_OMNI, utils.OMNI_DISPLAY)]
    for k, lbl in _consistency_methods:
        try:
            path = inter_ds_bed(ds, k)
            if os.path.exists(path):
                method_segs[lbl].append(match.load_bed(path))
        except Exception as e:
            print(f"Warning: could not load {lbl} for {ds}: {e}")

# 2. Compute consistency
method_counts = {}
for method_name, segs_list in method_segs.items():
    if not segs_list: continue
    cache_path = f"out/consistency/{method_name.lower()}.pkl"
    method_counts[method_name] = utils.cached_pickle(
        cache_path,
        lambda: analyze.compute_state_consistency(segs_list, bin_size=CHROMHMM_BIN, show_progress=True),
        label=f"consistency for {method_name}"
    )


### 2. Plotting

In [ ]:
# Use colors from the first reference dataset if possible
ref_colors = {}
if CHIP_DATASETS:
    for ds in CHIP_DATASETS:
        rp = ref_bed_path(ds)
        if os.path.exists(rp):
            ref_colors = match.state_colors(match.load_bed(rp))
            break

for method_name, counts in method_counts.items():
    print(f"--- Plotting {method_name} ---")
    out_path = f"out/consistency/{method_name.lower()}_consistency.png"
    analyze.plot_state_consistency(counts, method_name, out_path, colors=ref_colors)


### 3. Display

In [ ]:
for method_name in ["Reference", utils.CHROMHMM_DISPLAY, utils.HOMER_DISPLAY,
                    utils.MACS2_DISPLAY, utils.OMNI_DISPLAY]:
    img_path = f"out/consistency/{method_name.lower()}_consistency.png"
    show(img_path, caption=method_name)


# Joint models across replicates

`process_encode.sh` fits one model per dataset over both of its replicates — a joint
KMeans per peak caller and a joint ChromHMM over the concatenated binarized signal —
so `rep1` and `rep2` share a single state space, and then relabels both of them to the
ENCODE reference with one shared mapping: the joint state space survives the matching,
and joint and individual segmentations live in the same label space.

The section analyzes every joint segmentation the way the cells above analyze the
individual ones, compares the two replicates of every method, and answers:
1. **Fragmentation and entropy** — is one model over both replicates simpler than two
   independently learned ones?
2. **Replicate consistency** — do two independently learned models agree across the
   replicates as well as one model learned over both of them, overall and per state?
3. **Individual vs joint** — how much does a replicate's own segmentation change when
   the states are learned jointly?

Kappa and Jaccard come in two modes: **Full** over every state, and **NOQH** with the
Quies/Het bulk of the genome dropped, since agreeing on it is easy and otherwise
dominates both metrics.

### 1. Compute

In [ ]:
# 1. Compute: analyze every joint segmentation, then compare the individual
# and the joint segmentation of both replicates, per dataset.
JOINT_OUT = f"{SP}/joint"
os.makedirs(JOINT_OUT, exist_ok=True)

REPS = ["rep1", "rep2"]
# Every method of the replicate analysis, each individual before its joint counterpart.
REP_METHODS = INTER_DS_METHODS + JOINT_METHODS
REP_METHOD_LABELS = [utils.display_name(m) for m in REP_METHODS]
# (individual, joint) counterparts of the same binarization.
JOINT_INDIV_PAIRS = list(zip(INTER_DS_METHODS, JOINT_METHODS))


def joint_dir(ds):
    """Directory holding the replicate comparison of one dataset."""
    return f"out/{ds}/{MATCH_METHOD}/joint"


def joint_inputs(folder, method):
    """Binarized signal behind a joint segmentation, for its emissions.

    A joint model is learned over both replicates but segments each of them on
    its own, so the emissions of a replicate come from that replicate's signal.
    """
    if method == utils.JOINT_CHROMHMM:
        return [f"{folder}/{utils.CHROMHMM_DEFAULT}/*.txt"]
    caller = method.replace("joint_kmeans_", "")
    return [f"{folder}/{caller}/chromhmm_peaks/chr*.txt.gz"]


def analyze_joint_segmentations():
    """run_analyze per joint segmentation, one per replicate and method."""
    futures = {}
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        for ds in REP_DATASETS:
            cfg = DATASETS[ds]
            ds_annotations = list(annotations)
            if cfg.get("atac"):
                ds_annotations.append(f"{ds}/atac_{cfg['atac']}.bed.gz")

            for rep in REPS:
                folder = f"{ds}/{rep}"
                for method in JOINT_METHODS:
                    seg = inter_ds_bed(folder, method)
                    outdir = f"out/{folder}/{MATCH_METHOD}/{method}"
                    if not os.path.exists(seg):
                        print(f"  SKIP {method} ({rep}) for {ds}: no {seg}")
                        continue
                    if not needs_enrichment(outdir, cfg):
                        continue
                    print(f"Analyzing {method} ({rep}) for {ds}...")
                    fut = executor.submit(
                        analyze.run_analyze,
                        seg=seg, bin_size=seg_bin(seg),
                        outdir=outdir,
                        inputs=joint_inputs(folder, method),
                        annotations=ds_annotations,
                        bw_emissions=seg.replace(".bed", ".bw_emissions.npz"),
                        **rnaseq_args(ds, cfg),
                        emissions_only=False,
                    )
                    futures[fut] = f"{method} ({rep}) for {ds}"

    for fut in as_completed(futures):
        try:
            fut.result()
        except Exception as e:
            print(f"  ERROR: {futures[fut]}: {e}")


def compare_replicate_segmentations():
    """Compare rep1 against rep2 for every individual and joint method.

    all_pairs=False keeps the same-method rep1-vs-rep2 pairs the plots below read;
    the entropy and segment statistics cover every segmentation regardless. The
    stamp next to the results holds the segmentations that produced them, so a
    re-run only recomputes a dataset whose segmentations changed.
    """
    for ds in REP_DATASETS:
        segs = existing([inter_ds_bed(f"{ds}/{rep}", method)
                         for method in REP_METHODS for rep in REPS])
        if len(segs) < 2:
            print(f"  SKIP {ds}: {len(segs)} replicate segmentation(s) on disk")
            continue

        outdir = joint_dir(ds)
        stamp = f"{outdir}/compare_stamp.json"
        signature = {"segs": {p: utils.file_stamp(p) for p in segs}}
        outputs = [f"{outdir}/{name}" for name in
                   ("segment_stats.tsv", "entropy_summary.tsv", "per_state_metrics.tsv")]
        if utils.stamp_current(stamp, signature, outputs):
            print(f"  {ds} replicate comparison is up to date, skipped.")
            continue

        print(f"  {ds}: comparing {len(segs)} replicate segmentations ...")
        compare.run_compare(seg=segs, bins=[seg_bin(p) for p in segs], outdir=outdir)
        utils.save_stamp(stamp, signature)


analyze_joint_segmentations()
compare_replicate_segmentations()

In [ ]:
# 1b. The tables the plots read: aggregated per dataset, plus the agreement of
# the segmentation pairs, which is computed from the BEDs and therefore cached.
# (metrics key, display name) of the two modes every agreement is measured in.
MODES = [(utils.FULL, utils.FULL_DISPLAY), (utils.NOQH, utils.NOQH_DISPLAY)]


def load_replicate_tsv(filename):
    """Concatenated {joint_dir}/<filename> over the datasets with replicates.

    Splits the segmentation label of every row into its method and replicate.
    """
    frames = [pd.read_csv(f"{joint_dir(ds)}/{filename}", sep="\t").assign(Dataset=ds)
              for ds in REP_DATASETS if os.path.exists(f"{joint_dir(ds)}/{filename}")]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    keys = df["segmentation"].map(lambda label: label.rsplit("_", 1))
    df["Method"] = keys.map(lambda k: utils.display_name(k[0]))
    df["Rep"] = keys.map(lambda k: k[1])
    return df[df["Method"].isin(REP_METHOD_LABELS)]


def load_replicate_per_state():
    """Per-state rep1-vs-rep2 metrics of every method, over the same datasets."""
    rows = []
    for ds in REP_DATASETS:
        path = f"{joint_dir(ds)}/per_state_metrics.tsv"
        if not os.path.exists(path):
            continue
        for _, row in pd.read_csv(path, sep="\t").iterrows():
            method1, rep1 = str(row["seg1"]).rsplit("_", 1)
            method2, rep2 = str(row["seg2"]).rsplit("_", 1)
            if method1 != method2 or {rep1, rep2} != {"rep1", "rep2"}:
                continue
            rows.append({"Dataset": ds, "Method": utils.display_name(method1),
                         "State": row["state"],
                         utils.KAPPA_DISPLAY: row[utils.KAPPA],
                         utils.JACCARD_DISPLAY: row[utils.JACCARD]})
    return pd.DataFrame(rows)


def agreement(path1, path2):
    """Agreement of two segmentations per mode, None when they cannot be compared.

    Both modes come out of a single pass over the pair: Full keeps every state,
    NOQH drops the Quies/Het background. A pair that is not on disk, shares no
    state or never overlaps is reported as None and left out of the plots, rather
    than entering them as a row of zeros.
    """
    if not (os.path.exists(path1) and os.path.exists(path2)):
        return None
    s1, s2 = match.load_bed(path1), match.load_bed(path2)
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    overlap = match.pair_overlap(s1, s2)
    if not (set(l1) | set(l2)) or sum(overlap.values()) == 0:
        return None
    return match.agreement_by_mode(overlap, l1, l2, background=utils.NOQH_STATES)


def agreement_table(pairs):
    """DataFrame of agreement() over (method, dataset, path1, path2) tuples.

    One row per pair and mode, so a plot selects its mode by filtering on Mode.
    """
    rows = []
    for method, ds, path1, path2 in pairs:
        modes = agreement(path1, path2)
        if modes is None:
            print(f"  SKIP {ds} {method}: {path1} vs {path2}")
            continue
        print(f"  {ds} {method}: " + ", ".join(
            f"{name} {utils.KAPPA}={modes[key][utils.KAPPA]:.3f} {utils.JACCARD}={modes[key][utils.JACCARD]:.3f}"
            for key, name in MODES))
        for key, name in MODES:
            rows.append({"Method": utils.display_name(method), "Dataset": ds, "Mode": name,
                         **{utils.metric_display(m): v for m, v in modes[key].items()}})
    return pd.DataFrame(rows)


def compute_joint_rep_agreement():
    """rep1 vs rep2 of every method: two individual models against one joint model."""
    return agreement_table([
        (method, ds,
         inter_ds_bed(f"{ds}/rep1", method), inter_ds_bed(f"{ds}/rep2", method))
        for ds in REP_DATASETS
        for method in REP_METHODS])


def compute_joint_indiv_agreement():
    """Every replicate's own segmentation against its joint counterpart."""
    return agreement_table([
        (indiv, f"{ds}/{rep}",
         inter_ds_bed(f"{ds}/{rep}", indiv), inter_ds_bed(f"{ds}/{rep}", joint))
        for ds in REP_DATASETS
        for rep in REPS
        for indiv, joint in JOINT_INDIV_PAIRS])


def has_modes(df):
    """Cache guard: an agreement table holds both modes of every pair it kept.

    Also rejects a cache from before the modes were added, and an empty one,
    which means the segmentations were not on disk yet.
    """
    return not df.empty and set(df.get("Mode", [])) == {name for _, name in MODES}


# Read straight off the tables written above, so they follow a re-run.
df_joint_stats = load_replicate_tsv("segment_stats.tsv")
df_joint_entropy = load_replicate_tsv("entropy_summary.tsv")
df_joint_state = load_replicate_per_state()

df_joint_rep = utils.cached_pickle(
    "out/df_joint_rep.pkl", compute_joint_rep_agreement,
    label="replicate consistency, individual and joint", valid=has_modes)
df_joint_indiv = utils.cached_pickle(
    "out/df_joint_indiv.pkl", compute_joint_indiv_agreement,
    label="individual vs joint agreement", valid=has_modes)

### 2. Plotting

In [ ]:
# 2. Plotting: one bar per method, mean ± SE across datasets, every observation shown.
def _hatch_joint(ax, shown):
    """Hatch the bars of the joint models, which share their caller's colour."""
    for container, (method_key, _) in zip(ax.containers, shown):
        if method_key.startswith("joint_"):
            for bar in container:
                bar.set_hatch("//")


def _methods_present(df, methods):
    """(method key, display name) of every method with data, in the given order."""
    present = set(df["Method"])
    return [(m, utils.display_name(m)) for m in methods
            if utils.display_name(m) in present]


def _annotate_means(ax, df, x, order, metric, fmt):
    """Print the mean of every bar above it."""
    yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
    for i, name in enumerate(order):
        vals = df[df[x] == name][metric]
        if vals.empty:
            continue
        ax.text(i, vals.max() + yrange * 0.01, fmt.format(vals.mean()),
                ha="center", va="bottom", fontsize=6)


def joint_barplot(df, metric, methods, title, outfile,
                  ylabel=None, ylim=(0, 1.05), fmt="{:.2f}"):
    """Bar plot of one metric per method; methods without data are dropped."""
    if df.empty:
        print(f"  SKIP {outfile}: no data")
        return
    shown = _methods_present(df, methods)
    if not shown:
        print(f"  SKIP {outfile}: none of {methods} present")
        return
    order = [label for _, label in shown]

    fig, ax = plt.subplots(figsize=(max(7, len(order) * 1.3), 5))
    sns.barplot(data=df, x="Method", y=metric, order=order,
                hue="Method", hue_order=order, dodge=False, legend=False,
                palette={label: utils.method_color(m) for m, label in shown},
                capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
                edgecolor="lightgrey", linewidth=1, ax=ax)
    _hatch_joint(ax, shown)
    utils.strip_points(ax, data=df, x="Method", y=metric, order=order, size=3)
    ax.set_title(f"{title} (n={df['Dataset'].nunique()})", fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel or metric, fontsize=9)
    if ylim:
        # A NOQH kappa can go negative; keep such a bar inside the frame.
        ax.set_ylim(min(ylim[0], float(df[metric].min())), ylim[1])
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    _annotate_means(ax, df, "Method", order, metric, fmt)
    utils.save_fig(fig, outfile)


def joint_state_barplot(df, metric, methods, title, outfile):
    """Per-state bar plot of one metric, one bar per method within each state."""
    if df.empty:
        print(f"  SKIP {outfile}: no data")
        return
    shown = _methods_present(df, methods)
    if not shown:
        print(f"  SKIP {outfile}: none of {methods} present")
        return
    order = [label for _, label in shown]
    states = summary_plots.sort_states(df["State"].unique())

    fig, ax = plt.subplots(figsize=(max(10, len(states) * 0.9), 5))
    sns.barplot(data=df, x="State", y=metric, order=states,
                hue="Method", hue_order=order,
                palette={label: utils.method_color(m) for m, label in shown},
                capsize=0.05, errorbar="se", err_kws={"linewidth": 1.0},
                edgecolor="lightgrey", linewidth=0.5, ax=ax)
    _hatch_joint(ax, shown)
    utils.strip_points(ax, data=df, x="State", y=metric,
                       hue="Method", order=states, hue_order=order)
    ax.set_title(f"{title} (n={df['Dataset'].nunique()})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Chromatin state", fontsize=9)
    ax.set_ylabel(f"Replicate {metric}", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
    utils.save_fig(fig, outfile)


joint_barplot(df_joint_stats, "n_segments", REP_METHODS,
              "Number of segments per replicate: individual vs joint models",
              f"{JOINT_OUT}/joint_n_segments.png",
              ylabel="Number of segments", ylim=None, fmt="{:.0f}")
joint_barplot(df_joint_entropy, "total_entropy", REP_METHODS,
              "Transition matrix entropy per replicate: individual vs joint models",
              f"{JOINT_OUT}/joint_entropy.png",
              ylabel="Entropy (bits)", ylim=None)

for _metric, _label in ((utils.KAPPA, utils.KAPPA_DISPLAY),
                        (utils.JACCARD, utils.JACCARD_DISPLAY)):
    # Full over every state, NOQH without the Quies/Het background.
    for _key, _mode in MODES:
        _suffix = "" if _key == utils.FULL else f"_{_key}"
        joint_barplot(df_joint_rep[df_joint_rep["Mode"] == _mode], _label, REP_METHODS,
                      f"Replicate consistency ({_label}, {_mode}): individual vs joint models",
                      f"{JOINT_OUT}/joint_rep_consistency_{_metric}{_suffix}.png",
                      ylabel=f"{_label} ({_mode})")
        joint_barplot(df_joint_indiv[df_joint_indiv["Mode"] == _mode], _label, INTER_DS_METHODS,
                      f"Individual vs joint segmentation of a replicate ({_label}, {_mode})",
                      f"{JOINT_OUT}/joint_indiv_{_metric}{_suffix}.png",
                      ylabel=f"{_label} ({_mode})")
    joint_state_barplot(df_joint_state, _label, REP_METHODS,
                        f"Per-state replicate consistency ({_label})",
                        f"{JOINT_OUT}/joint_rep_per_state_{_metric}.png")

### 3. Display

In [ ]:
# 3. Display
show_group("Joint models across replicates — segments and entropy", [
    (f"{JOINT_OUT}/joint_n_segments.png",
     "Number of segments per replicate segmentation"),
    (f"{JOINT_OUT}/joint_entropy.png",
     "Transition matrix entropy per replicate segmentation"),
], level=2)

show_group("Joint models across replicates — replicate consistency", [
    (f"{JOINT_OUT}/joint_rep_consistency_kappa.png",
     f"Cohen's {utils.KAPPA_DISPLAY}, two individual models vs one joint model — {utils.FULL_DISPLAY}"),
    (f"{JOINT_OUT}/joint_rep_consistency_kappa_noqh.png",
     f"Cohen's {utils.KAPPA_DISPLAY}, two individual models vs one joint model — {utils.NOQH_DISPLAY} (excl. Quies/Het)"),
    (f"{JOINT_OUT}/joint_rep_consistency_jaccard.png",
     f"{utils.JACCARD_DISPLAY}, two individual models vs one joint model — {utils.FULL_DISPLAY}"),
    (f"{JOINT_OUT}/joint_rep_consistency_jaccard_noqh.png",
     f"{utils.JACCARD_DISPLAY}, two individual models vs one joint model — {utils.NOQH_DISPLAY} (excl. Quies/Het)"),
    (f"{JOINT_OUT}/joint_rep_per_state_kappa.png",
     f"Per-state replicate consistency (Cohen's {utils.KAPPA_DISPLAY})"),
    (f"{JOINT_OUT}/joint_rep_per_state_jaccard.png",
     f"Per-state replicate consistency ({utils.JACCARD_DISPLAY})"),
], level=2)

show_group("Joint models across replicates — individual vs joint", [
    (f"{JOINT_OUT}/joint_indiv_kappa.png",
     f"Individual vs joint segmentation of the same replicate: Cohen's {utils.KAPPA_DISPLAY} — {utils.FULL_DISPLAY}"),
    (f"{JOINT_OUT}/joint_indiv_kappa_noqh.png",
     f"Individual vs joint segmentation of the same replicate: Cohen's {utils.KAPPA_DISPLAY} — {utils.NOQH_DISPLAY}"),
    (f"{JOINT_OUT}/joint_indiv_jaccard.png",
     f"Individual vs joint segmentation of the same replicate: {utils.JACCARD_DISPLAY} — {utils.FULL_DISPLAY}"),
    (f"{JOINT_OUT}/joint_indiv_jaccard_noqh.png",
     f"Individual vs joint segmentation of the same replicate: {utils.JACCARD_DISPLAY} — {utils.NOQH_DISPLAY}"),
], level=2)

# Uncomment to show the per-segmentation analysis of the joint models
# for ds in REP_DATASETS:
#     for method in JOINT_METHODS:
#         show_group(f"{DS_TITLE.get(ds, ds)} — {utils.display_name(method)} (rep1)", [
#             (method_plot(f"{ds}/rep1", method, "enrichment/enrichment.png", MATCH_METHOD),
#              "functional enrichment"),
#             (method_plot(f"{ds}/rep1", method, "bin_emissions/state_emissions.png", MATCH_METHOD),
#              "binarized emissions"),
#             (method_plot(f"{ds}/rep1", method, "segment_length.png", MATCH_METHOD),
#              "segment length distribution"),
#         ], width=760, level=4)
#     break  # Comment to plot all the datasets